In [ ]:
# imports..

import torch
import torch.nn as nn
import torch.nn.functional as F

import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18

import numpy as np
import matplotlib.pyplot as plt

from tqdm import tqdm

In [ ]:
# device setup..

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

In [ ]:
# Dataset loading..

mean = (0.4914, 0.4822, 0.4465)
std = (0.247, 0.243, 0.261)

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

test_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=False,
    transform=test_transform
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=2
)

classes = test_dataset.classes

print("Test Samples:", len(test_dataset))

In [ ]:
# Gausian Blur..

class GaussianBlur(nn.Module):

    def __init__(self, channels):

        super(GaussianBlur, self).__init__()

        kernel = torch.tensor([
            [1., 2., 1.],
            [2., 4., 2.],
            [1., 2., 1.]
        ])

        kernel /= 16.0

        kernel = kernel.view(1, 1, 3, 3)

        kernel = kernel.repeat(channels, 1, 1, 1)

        self.weight = nn.Parameter(
            kernel,
            requires_grad=False
        )

        self.groups = channels

    def forward(self, x):

        return F.conv2d(
            x,
            self.weight,
            padding=1,
            groups=self.groups
        )

In [ ]:
# Denoise Block..

class DenoiseBlock(nn.Module):

    def __init__(self, channels):

        super(DenoiseBlock, self).__init__()

        self.conv1 = nn.Conv2d(
            channels,
            channels,
            3,
            padding=1
        )

        self.bn1 = nn.BatchNorm2d(channels)

        self.conv2 = nn.Conv2d(
            channels,
            channels,
            3,
            padding=1
        )

        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):

        residual = x

        out = F.relu(self.bn1(self.conv1(x)))

        out = self.bn2(self.conv2(out))

        out += residual

        out = F.relu(out)

        return out

In [ ]:
#AvgMAxPool..

class AvgMaxPool(nn.Module):

    def __init__(self):

        super(AvgMaxPool, self).__init__()

        self.avgpool = nn.AdaptiveAvgPool2d(1)

        self.maxpool = nn.AdaptiveMaxPool2d(1)

    def forward(self, x):

        avg = self.avgpool(x)

        max_ = self.maxpool(x)

        return avg + max_

In [ ]:
#secureRestnet18..

class SecureResNet18(nn.Module):

    def __init__(self, num_classes=10):

        super(SecureResNet18, self).__init__()

        self.backbone = resnet18(weights=None)

        self.backbone.conv1 = nn.Conv2d(
            3,
            64,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False
        )

        self.backbone.maxpool = nn.Identity()

        self.gaussian = GaussianBlur(64)

        self.denoise = DenoiseBlock(512)

        self.pool = AvgMaxPool()

        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):

        x = self.backbone.conv1(x)

        x = self.backbone.bn1(x)

        x = self.backbone.relu(x)

        x = self.gaussian(x)

        x = self.backbone.layer1(x)

        x = self.backbone.layer2(x)

        x = self.backbone.layer3(x)

        x = self.backbone.layer4(x)

        x = self.denoise(x)

        x = self.pool(x)

        x = torch.flatten(x, 1)

        x = self.fc(x)

        return x

In [ ]:
#load trained model..

model = SecureResNet18().to(device)

model.load_state_dict(
    torch.load(
        './best_securecnn.pth',
        map_location=device
    )
)

model.eval()

print("SecureCNN Loaded Successfully")

In [ ]:
# Clean Accuracy Evaluation..

correct = 0
total = 0

with torch.no_grad():

    for images, labels in tqdm(test_loader):

        images = images.to(device)

        labels = labels.to(device)

        outputs = model(images)

        _, predicted = outputs.max(1)

        total += labels.size(0)

        correct += predicted.eq(labels).sum().item()

clean_acc = 100. * correct / total

print(f"Clean Accuracy: {clean_acc:.2f}%")

# FGSM ATTACk FUnction...

In [ ]:
def fgsm_attack(model,
                images,
                labels,
                eps):

    images = images.clone().detach().to(device)

    labels = labels.to(device)

    images.requires_grad = True

    outputs = model(images)

    loss = nn.CrossEntropyLoss()(outputs, labels)

    model.zero_grad()

    loss.backward()

    grad = images.grad.data

    adv_images = images + eps * grad.sign()

    adv_images = torch.clamp(
        adv_images,
        min=-1,
        max=1
    )

    return adv_images

In [ ]:
# FGSM Evaluation Function..

def evaluate_fgsm(model,
                  test_loader,
                  eps):

    model.eval()

    correct = 0

    total = 0

    for images, labels in tqdm(test_loader):

        images = images.to(device)

        labels = labels.to(device)

        adv_images = fgsm_attack(
            model,
            images,
            labels,
            eps
        )

        outputs = model(adv_images)

        _, predicted = outputs.max(1)

        total += labels.size(0)

        correct += predicted.eq(labels).sum().item()

    acc = 100. * correct / total

    return acc

In [ ]:
# Run FGSM Evaluation..

fgsm_epsilons = [
    0.01,
    0.03,
    0.05,
    0.07
]

fgsm_results = []

for eps in fgsm_epsilons:

    acc = evaluate_fgsm(
        model,
        test_loader,
        eps
    )

    fgsm_results.append(acc)

    print(f"FGSM eps={eps}")
    print(f"Accuracy: {acc:.2f}%\n")

# PGD ATTACK FUNCTION

In [ ]:
def pgd_attack(model,
               images,
               labels,
               eps=8/255,
               alpha=2/255,
               steps=10):

    images = images.clone().detach().to(device)

    labels = labels.to(device)

    loss = nn.CrossEntropyLoss()

    adv_images = images.clone().detach()

    adv_images = adv_images + torch.empty_like(
        adv_images
    ).uniform_(-eps, eps)

    adv_images = torch.clamp(
        adv_images,
        -1,
        1
    )

    for _ in range(steps):

        adv_images.requires_grad = True

        outputs = model(adv_images)

        cost = loss(outputs, labels)

        grad = torch.autograd.grad(
            cost,
            adv_images,
            retain_graph=False,
            create_graph=False
        )[0]

        adv_images = adv_images.detach() + alpha * grad.sign()

        delta = torch.clamp(
            adv_images - images,
            min=-eps,
            max=eps
        )

        adv_images = torch.clamp(
            images + delta,
            min=-1,
            max=1
        ).detach()

    return adv_images

In [ ]:
# PGD Evaluation Function..

def evaluate_pgd(model,
                 test_loader,
                 eps,
                 alpha,
                 steps):

    model.eval()

    correct = 0

    total = 0

    for images, labels in tqdm(test_loader):

        images = images.to(device)

        labels = labels.to(device)

        adv_images = pgd_attack(
            model,
            images,
            labels,
            eps,
            alpha,
            steps
        )

        outputs = model(adv_images)

        _, predicted = outputs.max(1)

        total += labels.size(0)

        correct += predicted.eq(labels).sum().item()

    acc = 100. * correct / total

    return acc

In [ ]:
# Run PGD Evaluation..

pgd_epsilons = [
    0.01,
    0.03,
    0.05,
    0.07
]

pgd_results = []

for eps in pgd_epsilons:

    acc = evaluate_pgd(
        model,
        test_loader,
        eps=eps,
        alpha=eps/4,
        steps=10
    )

    pgd_results.append(acc)

    print(f"PGD eps={eps}")
    print(f"Accuracy: {acc:.2f}%\n")

In [ ]:
# STORE RESULTS
clean_acc = 90.24

fgsm_eps = [0.01, 0.03, 0.05, 0.07]

fgsm_secure = [
    83.20,
    71.65,
    61.49,
    54.02
]

fgsm_baseline = [
    43.15,
    17.40,
    11.62,
    8.36
]

pgd_eps = [0.01, 0.03, 0.05, 0.07]

pgd_secure = [
    79.99,
    59.14,
    40.30,
    27.15
]

pgd_baseline = [
    40.06,
    9.40,
    2.78,
    1.02
]

In [ ]:
# FGSM ROBUSTNESS CURVE

plt.figure(figsize=(8,5))

plt.plot(
    fgsm_eps,
    fgsm_baseline,
    marker='o',
    linewidth=3,
    label='Baseline CNN'
)

plt.plot(
    fgsm_eps,
    fgsm_secure,
    marker='s',
    linewidth=3,
    label='SecureCNN'
)

plt.xlabel("Epsilon")

plt.ylabel("Accuracy (%)")

plt.title("FGSM Robustness Comparison")

plt.legend()

plt.grid(True)

plt.show()

In [ ]:
# PGD ROBUSTNESS CURVE..

plt.figure(figsize=(8,5))

plt.plot(
    pgd_eps,
    pgd_baseline,
    marker='o',
    linewidth=3,
    label='Baseline CNN'
)

plt.plot(
    pgd_eps,
    pgd_secure,
    marker='s',
    linewidth=3,
    label='SecureCNN'
)

plt.xlabel("Epsilon")

plt.ylabel("Accuracy (%)")

plt.title("PGD Robustness Comparison")

plt.legend()

plt.grid(True)

plt.show()

In [ ]:
# CLEAN vs FGSM vs PGD BAR CHART..

labels = [
    "Clean",
    "FGSM",
    "PGD"
]

baseline_values = [
    74.90,
    17.40,
    9.40
]

secure_values = [
    90.24,
    71.65,
    59.14
]

x = np.arange(len(labels))

width = 0.35

plt.figure(figsize=(8,5))

plt.bar(
    x - width/2,
    baseline_values,
    width,
    label='Baseline CNN'
)

plt.bar(
    x + width/2,
    secure_values,
    width,
    label='SecureCNN'
)

plt.xticks(x, labels)

plt.ylabel("Accuracy (%)")

plt.title("Clean vs Adversarial Accuracy")

plt.legend()

plt.show()

In [ ]:
# ACCURACY DROP %..

baseline_drop = [
    74.90 - 43.15,
    74.90 - 17.40,
    74.90 - 11.62,
    74.90 - 8.36
]

secure_drop = [
    90.24 - 83.20,
    90.24 - 71.65,
    90.24 - 61.49,
    90.24 - 54.02
]

plt.figure(figsize=(8,5))

plt.plot(
    fgsm_eps,
    baseline_drop,
    marker='o',
    linewidth=3,
    label='Baseline CNN Drop'
)

plt.plot(
    fgsm_eps,
    secure_drop,
    marker='s',
    linewidth=3,
    label='SecureCNN Drop'
)

plt.xlabel("Epsilon")

plt.ylabel("Accuracy Drop (%)")

plt.title("Accuracy Degradation Under FGSM")

plt.legend()

plt.grid(True)

plt.show()

In [ ]:
#!pip install --upgrade numpy scikit-learn
from sklearn.metrics import confusion_matrix
import seaborn as sns

In [ ]:
# Generate Prdictions..

all_preds = []

all_labels = []

model.eval()

with torch.no_grad():

    for images, labels in tqdm(test_loader):

        images = images.to(device)

        outputs = model(images)

        _, predicted = outputs.max(1)

        all_preds.extend(
            predicted.cpu().numpy()
        )

        all_labels.extend(
            labels.numpy()
        )

In [ ]:
# Creating Confuion matrix..

cm = confusion_matrix(
    all_labels,
    all_preds
)

plt.figure(figsize=(10,8))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=classes,
    yticklabels=classes
)

plt.xlabel("Predicted")

plt.ylabel("True")

plt.title("SecureCNN Confusion Matrix")

plt.savefig(
    "confusion_matrix.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

# DEEPFOOL

In [ ]:
from art.attacks.evasion import DeepFool
from art.estimators.classification import PyTorchClassifier

In [ ]:
# Creating Art Classifier..

classifier = PyTorchClassifier(
    model=model,
    loss=nn.CrossEntropyLoss(),
    optimizer=None,
    input_shape=(3, 32, 32),
    nb_classes=10,
    clip_values=(-1, 1)
)

print("ART Classifier Ready")

In [ ]:
# Deepfool Attack..

deepfool_attack = DeepFool(
    classifier=classifier,
    max_iter=20,
    epsilon=1e-6
)

print("DeepFool Ready")

In [ ]:
# DeepFool Evaluation..

correct = 0

total = 0

for images, labels in tqdm(test_loader):

    images = images.numpy()

    labels = labels.numpy()

    adv_images = deepfool_attack.generate(x=images)

    adv_images = torch.tensor(
        adv_images
    ).float().to(device)

    labels = torch.tensor(
        labels
    ).to(device)

    outputs = model(adv_images)

    _, predicted = outputs.max(1)

    total += labels.size(0)

    correct += predicted.eq(labels).sum().item()

deepfool_acc = 100. * correct / total

print(f"DeepFool Accuracy: {deepfool_acc:.2f}%")